In [ ]:

# Comprehensive Proteomics Drug Screening Analysis - Notebook 3: Validation Drugs
# Pathway enrichment analysis for drugs with well-known mechanisms

import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

plt.style.use('science_machine')

print("Loading data...")
adata = ad.read_h5ad('corrected_filtered_data.h5ad')
de_results = pd.read_csv('de_results_all_drugs.csv')
drugs_meta = pd.read_csv('drugs.csv')

print(f"Data loaded: {adata.shape}, {de_results.shape[0]} DE results")


Loading data...


Data loaded: (1204, 7872), 1322496 DE results


In [ ]:

# Perform DE analysis for DFO (iron chelator) vs DMSO
print("Performing DE analysis for DFO...")
dfo_mask = adata.obs['sample_type'] == 'DFO'
dmso_mask = adata.obs['sample_type'] == 'Control'

dfo_data = adata.X[dfo_mask, :]
dmso_data = adata.X[dmso_mask, :]

dfo_results_data = []
for i, protein in enumerate(adata.var_names):
    t_stat, pval = ttest_ind(dfo_data[:, i], dmso_data[:, i])
    dfo_mean = np.mean(dfo_data[:, i])
    dmso_mean = np.mean(dmso_data[:, i])
    
    dfo_results_data.append({
        'protein': protein,
        'drug': 'deferoxamine',
        'log2fc': dfo_mean - dmso_mean,
        'pvalue': pval,
        'drug_mean': dfo_mean,
        'dmso_mean': dmso_mean,
        'n_drug_samples': dfo_data.shape[0],
        'n_dmso_samples': dmso_data.shape[0]
    })

dfo_de = pd.DataFrame(dfo_results_data)
_, dfo_de['fdr'], _, _ = multipletests(dfo_de['pvalue'], method='fdr_bh')
dfo_de['is_de'] = (dfo_de['fdr'] < 0.05) & (np.abs(dfo_de['log2fc']) >= 0.5)
dfo_de['direction'] = np.where(dfo_de['log2fc'] > 0, 'up', 'down')

print(f"DFO: {dfo_de['is_de'].sum()} DE proteins ({(dfo_de['is_de'] & (dfo_de['direction']=='up')).sum()} up, {(dfo_de['is_de'] & (dfo_de['direction']=='down')).sum()} down)")


Performing DE analysis for DFO...


DFO: 1740 DE proteins (402 up, 1338 down)


In [ ]:

# Define validation drugs with well-known mechanisms
validation_drugs = pd.DataFrame([
    {'drug': 'Simvastatin acid (ammonium)', 'known_mechanism': 'HMG-CoA reductase inhibition', 'target': 'HMG-CoA Reductase'},
    {'drug': 'Atorvastatin (hemicalcium salt)', 'known_mechanism': 'HMG-CoA reductase inhibition', 'target': 'HMG-CoA Reductase'},
    {'drug': 'Digitoxin', 'known_mechanism': 'Na+/K+ ATPase inhibition', 'target': 'Na+/K+ ATPase'},
    {'drug': 'Budesonide', 'known_mechanism': 'Glucocorticoid receptor agonist', 'target': 'Glucocorticoid Receptor'},
    {'drug': 'Betamethasone valerate', 'known_mechanism': 'Glucocorticoid receptor agonist', 'target': 'Glucocorticoid Receptor'},
    {'drug': 'Erlotinib (Hydrochloride)', 'known_mechanism': 'EGFR tyrosine kinase inhibition', 'target': 'EGFR'},
    {'drug': 'Haloperidol', 'known_mechanism': 'Dopamine D2 receptor antagonist', 'target': 'Dopamine Receptor'},
    {'drug': 'Betahistine', 'known_mechanism': 'Histamine H1 agonist/H3 antagonist', 'target': 'Histamine Receptor'},
    {'drug': 'Mobocertinib (succinate)', 'known_mechanism': 'EGFR tyrosine kinase inhibition', 'target': 'EGFR'},
    {'drug': 'deferoxamine', 'known_mechanism': 'Iron chelation', 'target': 'Iron binding'},
])

validation_drugs.to_csv('validation_drugs_list.csv', index=False)
print("Validation drugs:")
print(validation_drugs.to_string(index=False))


Validation drugs:
                           drug                    known_mechanism                  target
    Simvastatin acid (ammonium)       HMG-CoA reductase inhibition       HMG-CoA Reductase
Atorvastatin (hemicalcium salt)       HMG-CoA reductase inhibition       HMG-CoA Reductase
                      Digitoxin           Na+/K+ ATPase inhibition           Na+/K+ ATPase
                     Budesonide    Glucocorticoid receptor agonist Glucocorticoid Receptor
         Betamethasone valerate    Glucocorticoid receptor agonist Glucocorticoid Receptor
      Erlotinib (Hydrochloride)    EGFR tyrosine kinase inhibition                    EGFR
                    Haloperidol    Dopamine D2 receptor antagonist       Dopamine Receptor
                    Betahistine Histamine H1 agonist/H3 antagonist      Histamine Receptor
       Mobocertinib (succinate)    EGFR tyrosine kinase inhibition                    EGFR
                   deferoxamine                     Iron chelation      

In [ ]:

# Check what's going on
print("Type of validation_drugs:", type(validation_drugs))
print("Type of len:", type(len))
print("Type of range:", type(range))

# Try a simple loop
for i in [0, 1]:
    print(f"i = {i}")


Type of validation_drugs: <class 'pandas.core.frame.DataFrame'>
Type of len: <class 'builtin_function_or_method'>
Type of range: <class 'type'>
i = 0
i = 1


In [ ]:

# Try without the loop first - test with one drug
drug_name = 'Simvastatin acid (ammonium)'
print(f"Testing with {drug_name}")

de_proteins = de_results[(de_results['drug'] == drug_name) & (de_results['is_de'] == True)]['protein'].values
print(f"DE proteins: {len(de_proteins)}")
print(f"Type: {type(de_proteins)}")
print(f"First 5: {de_proteins[:5]}")

# Convert to list explicitly
gene_list_test = [str(g) for g in de_proteins]
print(f"Gene list length: {len(gene_list_test)}")
print(f"Type: {type(gene_list_test)}")


Testing with Simvastatin acid (ammonium)
DE proteins: 316
Type: <class 'numpy.ndarray'>
First 5: ['MED19' 'TARS3' 'EPOP' 'CCDC85C' 'CYSRT1']
Gene list length: 316
Type: <class 'list'>


In [ ]:

# Try enrichment with explicit list
try:
    enr_test = gp.enrichr(
        gene_list=gene_list_test,
        gene_sets='KEGG_2021_Human',
        organism='human',
        outdir=None,
        cutoff=0.05
    )
    print("Success!")
    print(f"Results: {enr_test.results.shape if enr_test.results is not None else 'None'}")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


Success!
Results: (213, 10)


In [ ]:

# Check for variable conflicts
import builtins
print("list is builtin list:", list is builtins.list)
print("Type of list:", type(list))

# Check all variables
all_vars = dir()
print("\nVariables containing 'list':")
for v in all_vars:
    if 'list' in v.lower():
        print(f"  {v}: {type(eval(v))}")


list is builtin list: True
Type of list: <class 'type'>

Variables containing 'list':
  _jupyterlab_variableinspector_dict_list: <class 'function'>
  gene_list_input: <class 'list'>
  gene_list_test: <class 'list'>


In [ ]:

# Initialize results storage
all_enr_results = []

# Drug 1: Simvastatin
drug = 'Simvastatin acid (ammonium)'
print(f"Processing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

print(f"Total results so far: {len(all_enr_results)}")


Processing Simvastatin acid (ammonium)...
  316 DE proteins


  KEGG_2021_Human: 213 pathways


  Reactome_2022: 722 pathways


  GO_Biological_Process_2023: 1829 pathways
Total results so far: 3


In [ ]:

# Drug 2: Atorvastatin
drug = 'Atorvastatin (hemicalcium salt)'
print(f"Processing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

# Drug 3: Digitoxin
drug = 'Digitoxin'
print(f"\nProcessing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

print(f"\nTotal results so far: {len(all_enr_results)}")


Processing Atorvastatin (hemicalcium salt)...
  61 DE proteins


  KEGG_2021_Human: 87 pathways


  Reactome_2022: 242 pathways


  GO_Biological_Process_2023: 559 pathways

Processing Digitoxin...
  2013 DE proteins


  KEGG_2021_Human: 300 pathways


  Reactome_2022: 1431 pathways


  GO_Biological_Process_2023: 4142 pathways

Total results so far: 9


In [ ]:

# Drug 4: Budesonide
drug = 'Budesonide'
print(f"Processing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

# Drug 5: Betamethasone valerate
drug = 'Betamethasone valerate'
print(f"\nProcessing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

print(f"\nTotal results so far: {len(all_enr_results)}")


Processing Budesonide...
  164 DE proteins


  KEGG_2021_Human: 145 pathways


  Reactome_2022: 451 pathways


  GO_Biological_Process_2023: 1168 pathways

Processing Betamethasone valerate...
  135 DE proteins


  KEGG_2021_Human: 131 pathways


  Reactome_2022: 460 pathways


  GO_Biological_Process_2023: 1128 pathways

Total results so far: 15


In [ ]:

# Drug 6: Erlotinib
drug = 'Erlotinib (Hydrochloride)'
print(f"Processing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

# Drug 7: Haloperidol
drug = 'Haloperidol'
print(f"\nProcessing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

# Drug 8: Betahistine
drug = 'Betahistine'
print(f"\nProcessing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

print(f"\nTotal results so far: {len(all_enr_results)}")


Processing Erlotinib (Hydrochloride)...
  872 DE proteins


  KEGG_2021_Human: 270 pathways


  Reactome_2022: 1080 pathways


  GO_Biological_Process_2023: 3072 pathways

Processing Haloperidol...
  364 DE proteins


  KEGG_2021_Human: 218 pathways


  Reactome_2022: 772 pathways


  GO_Biological_Process_2023: 2412 pathways

Processing Betahistine...
  66 DE proteins


  KEGG_2021_Human: 125 pathways


  Reactome_2022: 260 pathways


  GO_Biological_Process_2023: 461 pathways

Total results so far: 24


In [ ]:

# Drug 9: Mobocertinib
drug = 'Mobocertinib (succinate)'
print(f"Processing {drug}...")
de_prots = de_results[(de_results['drug'] == drug) & (de_results['is_de'] == True)]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

# Drug 10: deferoxamine (DFO)
drug = 'deferoxamine'
print(f"\nProcessing {drug}...")
de_prots = dfo_de[dfo_de['is_de'] == True]['protein'].tolist()
print(f"  {len(de_prots)} DE proteins")

for db in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
    enr = gp.enrichr(gene_list=de_prots, gene_sets=db, organism='human', outdir=None, cutoff=0.05)
    if enr.results is not None and len(enr.results) > 0:
        res = enr.results.copy()
        res['drug'] = drug
        res['gene_set'] = db
        all_enr_results.append(res)
        print(f"  {db}: {len(res)} pathways")

print(f"\n{'='*80}")
print(f"ENRICHMENT COMPLETE")
print(f"Total result dataframes: {len(all_enr_results)}")
print('='*80)


Processing Mobocertinib (succinate)...
  2907 DE proteins


  KEGG_2021_Human: 317 pathways


  Reactome_2022: 1596 pathways


  GO_Biological_Process_2023: 4572 pathways

Processing deferoxamine...
  1740 DE proteins


  KEGG_2021_Human: 311 pathways


  Reactome_2022: 1408 pathways


  GO_Biological_Process_2023: 3840 pathways

ENRICHMENT COMPLETE
Total result dataframes: 30


In [ ]:

# Combine all enrichment results
validation_enrichment = pd.concat(all_enr_results, ignore_index=True)

print(f"Combined enrichment results: {validation_enrichment.shape}")
print(f"\nColumns: {validation_enrichment.columns.tolist()}")
print(f"\nDrugs: {validation_enrichment['drug'].unique()}")
print(f"\nGene sets: {validation_enrichment['gene_set'].unique()}")

# Save results
validation_enrichment.to_csv('validation_enrichment_results.csv', index=False)
print("\nSaved validation_enrichment_results.csv")

# Summary statistics
print("\n" + "="*80)
print("ENRICHMENT SUMMARY")
print("="*80)
for drug in validation_drugs['drug']:
    drug_enr = validation_enrichment[validation_enrichment['drug'] == drug]
    print(f"{drug}: {len(drug_enr)} total pathways")
    for gs in ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']:
        n = len(drug_enr[drug_enr['gene_set'] == gs])
        print(f"  {gs}: {n}")


Combined enrichment results: (33722, 12)

Columns: ['Gene_set', 'Term', 'Overlap', 'P-value', 'Adjusted P-value', 'Old P-value', 'Old Adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes', 'drug', 'gene_set']

Drugs: ['Simvastatin acid (ammonium)' 'Atorvastatin (hemicalcium salt)'
 'Digitoxin' 'Budesonide' 'Betamethasone valerate'
 'Erlotinib (Hydrochloride)' 'Haloperidol' 'Betahistine'
 'Mobocertinib (succinate)' 'deferoxamine']

Gene sets: ['KEGG_2021_Human' 'Reactome_2022' 'GO_Biological_Process_2023']



Saved validation_enrichment_results.csv

ENRICHMENT SUMMARY
Simvastatin acid (ammonium): 2764 total pathways
  KEGG_2021_Human: 213
  Reactome_2022: 722
  GO_Biological_Process_2023: 1829
Atorvastatin (hemicalcium salt): 888 total pathways
  KEGG_2021_Human: 87
  Reactome_2022: 242
  GO_Biological_Process_2023: 559
Digitoxin: 5873 total pathways
  KEGG_2021_Human: 300
  Reactome_2022: 1431
  GO_Biological_Process_2023: 4142
Budesonide: 1764 total pathways
  KEGG_2021_Human: 145
  Reactome_2022: 451
  GO_Biological_Process_2023: 1168
Betamethasone valerate: 1719 total pathways
  KEGG_2021_Human: 131
  Reactome_2022: 460
  GO_Biological_Process_2023: 1128
Erlotinib (Hydrochloride): 4422 total pathways
  KEGG_2021_Human: 270
  Reactome_2022: 1080
  GO_Biological_Process_2023: 3072
Haloperidol: 3402 total pathways
  KEGG_2021_Human: 218
  Reactome_2022: 772
  GO_Biological_Process_2023: 2412
Betahistine: 846 total pathways
  KEGG_2021_Human: 125
  Reactome_2022: 260
  GO_Biological_Proces

In [ ]:

# Examine enrichment results for each validation drug to confirm known mechanisms

# 1. STATINS - Look for cholesterol/lipid metabolism pathways
print("="*80)
print("1. STATINS - Cholesterol/Lipid Metabolism")
print("="*80)

for drug in ['Simvastatin acid (ammonium)', 'Atorvastatin (hemicalcium salt)']:
    print(f"\n{drug}:")
    drug_enr = validation_enrichment[validation_enrichment['drug'] == drug]
    
    # Look for cholesterol/steroid/lipid pathways
    lipid_pathways = drug_enr[drug_enr['Term'].str.contains(
        'cholesterol|steroid|lipid|mevalonate|HMG', case=False, na=False
    )].sort_values('Adjusted P-value')
    
    print(f"  Found {len(lipid_pathways)} cholesterol/lipid-related pathways")
    print("\n  Top 10 pathways:")
    for idx, row in lipid_pathways.head(10).iterrows():
        print(f"    {row['Term'][:70]:<70} (FDR={row['Adjusted P-value']:.2e}, {row['Overlap']})")


1. STATINS - Cholesterol/Lipid Metabolism

Simvastatin acid (ammonium):
  Found 90 cholesterol/lipid-related pathways

  Top 10 pathways:
    Cholesterol Biosynthesis R-HSA-191273                                  (FDR=5.82e-11, 11/26)
    Cholesterol Biosynthetic Process (GO:0006695)                          (FDR=5.46e-08, 9/25)
    Regulation Of Cholesterol Biosynthesis By SREBP (SREBF) R-HSA-1655829  (FDR=2.00e-07, 11/55)
    Cholesterol Metabolic Process (GO:0008203)                             (FDR=2.48e-07, 12/67)
    Steroid biosynthesis                                                   (FDR=3.18e-06, 7/20)
    Metabolism Of Steroids R-HSA-8957322                                   (FDR=1.43e-04, 13/153)
    Metabolism Of Lipids R-HSA-556833                                      (FDR=2.41e-04, 30/732)
    Negative Regulation Of Cholesterol Transport (GO:0032375)              (FDR=3.07e-03, 4/10)
    Phospholipid Efflux (GO:0033700)                                       (FDR=3.07e-0

In [ ]:

# 2. DEFEROXAMINE - Look for iron/ferroptosis pathways
print("="*80)
print("2. DEFEROXAMINE - Iron Chelation/Ferroptosis")
print("="*80)

drug = 'deferoxamine'
drug_enr = validation_enrichment[validation_enrichment['drug'] == drug]

# Look for iron/ferroptosis pathways
iron_pathways = drug_enr[drug_enr['Term'].str.contains(
    'iron|ferr|metal|oxidative', case=False, na=False
)].sort_values('Adjusted P-value')

print(f"\nFound {len(iron_pathways)} iron/ferroptosis-related pathways")
print("\nTop 15 pathways:")
for idx, row in iron_pathways.head(15).iterrows():
    print(f"  {row['Term'][:70]:<70} (FDR={row['Adjusted P-value']:.2e}, {row['Overlap']})")

# Also look for hypoxia pathways (DFO is known to induce hypoxia-like response)
hypoxia_pathways = drug_enr[drug_enr['Term'].str.contains(
    'hypoxia|HIF|oxygen', case=False, na=False
)].sort_values('Adjusted P-value')

print(f"\nFound {len(hypoxia_pathways)} hypoxia-related pathways")
print("\nTop 10 hypoxia pathways:")
for idx, row in hypoxia_pathways.head(10).iterrows():
    print(f"  {row['Term'][:70]:<70} (FDR={row['Adjusted P-value']:.2e}, {row['Overlap']})")


2. DEFEROXAMINE - Iron Chelation/Ferroptosis

Found 40 iron/ferroptosis-related pathways

Top 15 pathways:
  Oxidative Phosphorylation (GO:0006119)                                 (FDR=2.88e-14, 31/63)
  Oxidative phosphorylation                                              (FDR=1.56e-08, 37/133)
  Mitochondrial Iron-Sulfur Cluster Biogenesis R-HSA-1362409             (FDR=8.69e-04, 7/13)
  Cytosolic Iron-Sulfur Cluster Assembly R-HSA-2564830                   (FDR=4.41e-03, 6/12)
  Metallo-Sulfur Cluster Assembly (GO:0031163)                           (FDR=6.56e-03, 10/27)
  Iron-Sulfur Cluster Assembly (GO:0016226)                              (FDR=6.56e-03, 10/27)
  Response To Metal Ions R-HSA-5660526                                   (FDR=1.01e-02, 6/14)
  Metallothioneins Bind Metals R-HSA-5661231                             (FDR=1.98e-02, 5/11)
  Protein Maturation By Iron-Sulfur Cluster Transfer (GO:0097428)        (FDR=3.27e-02, 6/14)
  Response To Metal Ion (GO:0010038)      

In [ ]:

# 3. DIGITOXIN - Look for Na+/K+ ATPase, cardiac, cell cycle pathways
print("="*80)
print("3. DIGITOXIN - Na+/K+ ATPase Inhibition")
print("="*80)

drug = 'Digitoxin'
drug_enr = validation_enrichment[validation_enrichment['drug'] == drug]

# Look for ion transport, cardiac, cell cycle pathways
cardiac_pathways = drug_enr[drug_enr['Term'].str.contains(
    'cardiac|heart|muscle contraction|ion transport|ATPase|sodium|potassium', case=False, na=False
)].sort_values('Adjusted P-value')

print(f"\nFound {len(cardiac_pathways)} cardiac/ion transport pathways")
print("\nTop 15 pathways:")
for idx, row in cardiac_pathways.head(15).iterrows():
    print(f"  {row['Term'][:70]:<70} (FDR={row['Adjusted P-value']:.2e}, {row['Overlap']})")

# Cell cycle (known effect of digitoxin)
cell_cycle_pathways = drug_enr[drug_enr['Term'].str.contains(
    'cell cycle|mitotic|G2|M phase', case=False, na=False
)].sort_values('Adjusted P-value')

print(f"\nFound {len(cell_cycle_pathways)} cell cycle pathways")
print("\nTop 10 cell cycle pathways:")
for idx, row in cell_cycle_pathways.head(10).iterrows():
    print(f"  {row['Term'][:70]:<70} (FDR={row['Adjusted P-value']:.2e}, {row['Overlap']})")


3. DIGITOXIN - Na+/K+ ATPase Inhibition

Found 110 cardiac/ion transport pathways

Top 15 pathways:
  Regulation Of Cardiac Muscle Cell Differentiation (GO:2000725)         (FDR=2.88e-01, 3/9)
  Cardiac Ventricle Morphogenesis (GO:0003208)                           (FDR=2.94e-01, 8/42)
  Cell Surface Receptor Signaling Pathway Involved In Heart Development  (FDR=3.40e-01, 4/16)
  Regulation Of Cardiac Muscle Cell Apoptotic Process (GO:0010665)       (FDR=3.40e-01, 3/10)
  Negative Regulation Of Cardiac Muscle Cell Differentiation (GO:2000726 (FDR=3.63e-01, 2/5)
  Iron Ion Transport (GO:0006826)                                        (FDR=3.66e-01, 5/24)
  RHOBTB3 ATPase Cycle R-HSA-9706019                                     (FDR=3.88e-01, 3/10)
  Cardiac Conduction System Development (GO:0003161)                     (FDR=3.94e-01, 5/25)
  Cardiac Atrium Morphogenesis (GO:0003209)                              (FDR=3.94e-01, 4/18)
  Cardiac Ventricle Development (GO:0003231)            

In [ ]:

# 4. GLUCOCORTICOIDS - Look for anti-inflammatory, immune response pathways
print("="*80)
print("4. GLUCOCORTICOIDS - Anti-inflammatory/Immune Response")
print("="*80)

for drug in ['Budesonide', 'Betamethasone valerate']:
    print(f"\n{drug}:")
    drug_enr = validation_enrichment[validation_enrichment['drug'] == drug]
    
    # Look for inflammatory/immune pathways
    immune_pathways = drug_enr[drug_enr['Term'].str.contains(
        'inflamm|immune|cytokine|NF-kappa|interferon|interleukin', case=False, na=False
    )].sort_values('Adjusted P-value')
    
    print(f"  Found {len(immune_pathways)} inflammatory/immune pathways")
    print("\n  Top 10 pathways:")
    for idx, row in immune_pathways.head(10).iterrows():
        print(f"    {row['Term'][:70]:<70} (FDR={row['Adjusted P-value']:.2e}, {row['Overlap']})")


4. GLUCOCORTICOIDS - Anti-inflammatory/Immune Response

Budesonide:
  Found 63 inflammatory/immune pathways

  Top 10 pathways:
    Cellular Response To Type II Interferon (GO:0071346)                   (FDR=2.88e-01, 3/66)
    Somatic Recombination Of Immunoglobulin Genes Involved In Immune Respo (FDR=2.91e-01, 1/9)
    Cellular Response To Type III Interferon (GO:0071358)                  (FDR=2.91e-01, 1/10)
    Type II Interferon-Mediated Signaling Pathway (GO:0060333)             (FDR=2.91e-01, 1/9)
    Neutrophil Activation Involved In Immune Response (GO:0002283)         (FDR=2.91e-01, 1/11)
    Negative Regulation Of Cytokinesis (GO:0032466)                        (FDR=2.91e-01, 1/5)
    Regulation Of Interleukin-1-Mediated Signaling Pathway (GO:2000659)    (FDR=2.91e-01, 1/5)
    Type III Interferon-Mediated Signaling Pathway (GO:0038196)            (FDR=2.91e-01, 1/10)
    Positive Regulation Of Interleukin-8 Production (GO:0032757)           (FDR=2.97e-01, 2/61)
    Cellular